# Sci/Tech News Q&A with Gemma (RAG) + Visual Analytics
**Track:** Build w Gemma — Gemma Challenge (Google Developer Group) · Hack Fiesta Miami, Aug 2026

This notebook implements a retrieval-augmented generation (RAG) pipeline on the **20 Newsgroups** dataset (`SetFit/20_newsgroups`), plus a scikit-learn analysis layer and five designed charts:

1. Load 20 Newsgroups from Hugging Face and keep only the **Sci/Tech** groups (`sci.*` and `comp.*`, 9 subtopics).
2. Clean, chunk and embed the posts with a small sentence-embedding model (metadata is kept per chunk).
3. Build a **FAISS** similarity index, with optional metadata filters (subtopic / group).
4. Use **scikit-learn** to measure how well the embedding space separates the subtopics (KMeans, Logistic Regression, PCA).
5. Draw a **Marimekko**, **dendrogram**, **stream graph**, **radial bar charts** and a **3D PCA volume plot** in one high-contrast palette with extruded (3D-look) shapes.
6. Feed retrieved context to Gemma (`google/gemma-2-2b-it`) and generate a cited answer.
7. Export the curated corpus, embeddings and FAISS index.

## 1. Install dependencies

In [1]:
!pip install -qq --only-binary=:all: transformers accelerate datasets sentence-transformers faiss-cpu bitsandbytes
!pip install -qq --only-binary=:all: pandas==2.2.2 matplotlib==3.8.4 scikit-learn==1.3.2 scipy==1.11.4 numpy==1.26.4

print("Environment dependencies normalized.")

ERROR: Ignored the following yanked versions: 3.0.4
ERROR: Could not find a version that satisfies the requirement pandas==2.2.2 (from versions: 2.2.3, 2.3.0, 2.3.1, 2.3.2, 2.3.3, 3.0.0rc0, 3.0.0rc1, 3.0.0rc2, 3.0.0, 3.0.1, 3.0.2, 3.0.3, 3.0.5, 3.0.6)
ERROR: No matching distribution found for pandas==2.2.2
Environment dependencies normalized.


## 2. Configuration

In [2]:
import re, json
import numpy as np
import pandas as pd
from datasets import load_dataset

SEED = 42
KEEP_PREFIXES = ("sci.", "comp.")
MAX_TRAIN_PER_TOPIC = 300
MAX_TEST_PER_TOPIC = 100
MIN_WORDS = 30
CHUNK_WORDS, CHUNK_OVERLAP, MAX_CHUNKS_PER_DOC = 120, 20, 4
EMBED_MODEL = "all-MiniLM-L6-v2"
GEMMA_MODEL_ID = "google/gemma-2-2b-it"

ds = load_dataset("SetFit/20_newsgroups")
print(ds)

NEWSGROUPS = ["comp.graphics", "comp.sys.ibm.pc.hardware",
              "comp.sys.mac.hardware", "comp.windows.x", "misc.forsale", "sci.crypt", "sci.electronics", "sci.med",
              "sci.space"]

def to_df(split):
    df = split.to_pandas()
    if "label_text" not in df.columns:
        df["label_text"] = df["label"].map(lambda i: NEWSGROUPS[i])
    return df

raw_train, raw_test = to_df(ds["train"]), to_df(ds["test"])
print("Raw rows:", len(raw_train), "train /", len(raw_test), "test")
print(raw_train.label_text.value_counts().head(20))

README.md:   0%|          | 0.00/734 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B / 14.8MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl: reconstructing file:   0%|          |  0.00B / 8.91MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11314 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7532 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 11314
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 7532
    })
})
Raw rows: 11314 train / 7532 test
label_text
rec.sport.hockey            600
soc.religion.christian      599
rec.motorcycles             598
rec.sport.baseball          597
sci.crypt                   595
sci.med                     594
rec.autos                   594
sci.space                   593
comp.windows.x              593
comp.os.ms-windows.misc     591
sci.electronics             591
comp.sys.ibm.pc.hardware    590
misc.forsale                585
comp.graphics               584
comp.sys.mac.hardware       578
talk.politics.mideast       564
talk.politics.guns          546
alt.atheism                 480
talk.politics.misc          465
talk.religion.misc          377
Name: count, dtype: int64


In [3]:
EMAIL = re.compile(r"\S+@\S+")

def make_title(raw):
    for line in raw.splitlines():
        s = line.strip()
        if (len(s) >= 15 and not s.startswith((">", "|", "-", "=", ":"))
                and "@" not in s and "writes:" not in s and not s.lower().startswith("in article")):
            return s[:90]
    return re.sub(r"\s+", " ", raw).strip()[:90]

def clean(text):
    t = re.sub(r"(?m)^\s*>.*$", " ", text)
    t = EMAIL.sub(" ", t)
    t = re.sub(r"[-=_*~#]{4,}", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def prepare(df, prefix, per_topic):
    df = df[df.label_text.str.startswith(KEEP_PREFIXES)].copy()
    df["title"] = [make_title(t) for t in df.text]
    df["clean"] = df.text.map(clean)
    df["n_words"] = df.clean.str.split().str.len()
    df = df[df.n_words >= MIN_WORDS]
    df = df.sample(frac=1, random_state=SEED).groupby("label_text", group_keys=False).head(per_topic)
    df["group"] = df.label_text.str.split(".").str[0]
    df = df.sort_values("label_text").reset_index(drop=True)
    df["doc_id"] = [f"{prefix}-{i}" for i in range(len(df))]
    return df[["doc_id", "label_text", "group", "title", "n_words", "clean"]]

train_df = prepare(raw_train, "train", MAX_TRAIN_PER_TOPIC)
test_df = prepare(raw_test, "test", MAX_TEST_PER_TOPIC)

# sci.* first, then comp.*
TOPICS = sorted(train_df.label_text.unique(), key=lambda t: (t.split(".")[0] != "sci", t))
print(f"{len(train_df)} train docs, {len(test_df)} test docs, {len(TOPICS)} subtopics: {TOPICS}\n")
print(train_df.groupby(["group", "label_text"]).n_words.agg(["count", "median", "max"]))
train_df.head(3)

2700 train docs, 900 test docs, 9 subtopics: ['sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x']

                                count  median   max
group label_text                                   
comp  comp.graphics               300    74.0  9103
      comp.os.ms-windows.misc     300    89.0  2362
      comp.sys.ibm.pc.hardware    300    96.0  1727
      comp.sys.mac.hardware       300    83.0   709
      comp.windows.x              300    95.5  7523
sci   sci.crypt                   300   145.5  5475
      sci.electronics             300   103.5   657
      sci.med                     300   113.0  4965
      sci.space                   300    94.5  6100


,doc_id,label_text,group,title,n_words,clean
0,train-0,comp.graphics,comp,"Well, i have a 386/40 with SVGA 1Mb. (OAK chip...",36,"Hi there!... Well, i have a 386/40 with SVGA 1..."
1,train-1,comp.graphics,comp,I am looking for a public domain 3d viewer. I...,37,I am looking for a public domain 3d viewer. It...
2,train-2,comp.graphics,comp,I'm not sure if this is the correct place to a...,66,I'm not sure if this is the correct place to a...


## 3. Chunk

In [4]:
from sentence_transformers import SentenceTransformer

def chunk_words(text, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP, max_chunks=MAX_CHUNKS_PER_DOC):
    words, step, out = text.split(), size - overlap, []
    for start in range(0, max(len(words), 1), step):
        piece = words[start:start + size]
        if len(piece) < 20 and out:
            break
        out.append(" ".join(piece))
        if len(out) >= max_chunks or start + size >= len(words):
            break
    return out

rows = []
for r in train_df.itertuples():
    for j, ch in enumerate(chunk_words(r.clean)):
        rows.append(dict(chunk_id=f"{r.doc_id}#{j}", doc_id=r.doc_id, chunk_no=j, title=r.title,
                         label_text=r.label_text, group=r.group, n_words=len(ch.split()), text=ch))
chunks_df = pd.DataFrame(rows)

embedder = SentenceTransformer(EMBED_MODEL)
embeddings = embedder.encode(chunks_df.text.tolist(), batch_size=128,
                             show_progress_bar=True, normalize_embeddings=True).astype("float32")
print("Chunks:", len(chunks_df), "| embedding matrix:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Chunks: 4476 | embedding matrix: (4476, 384)


## 5. FAISS index

In [5]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built with {index.ntotal} vectors of dimension {dim}.")

def retrieve(question, top_k=5, topic=None, group=None):
    """Top-k chunks; optionally restrict to one subtopic (e.g. 'sci.med') or group ('sci' / 'comp')."""
    q = embedder.encode([question], normalize_embeddings=True).astype("float32")
    fetch_k = index.ntotal if (topic or group) else top_k
    scores, idxs = index.search(q, fetch_k)
    hits = []
    for s, i in zip(scores[0], idxs[0]):
        row = chunks_df.iloc[i]
        if topic and row.label_text != topic: continue
        if group and row.group != group: continue
        hits.append(dict(title=row.title, text=row.text, topic=row.label_text, score=float(s)))
        if len(hits) == top_k: break
    return hits

for r in retrieve("orbital launch of a satellite", top_k=3):
    print(f"{r['score']:.3f}  [{r['topic']}]  {r['title']}")
print("--- filtered to sci.med ---")
for r in retrieve("treatment for headaches", top_k=3, topic="sci.med"):
    print(f"{r['score']:.3f}  [{r['topic']}]  {r['title']}")

FAISS index built with 4476 vectors of dimension 384.
0.435  [sci.space]  I would just like to point out that it is much easier to place an
0.412  [sci.space]  Ethnocentric USian that I am, I've assumed that we and the
0.405  [sci.space]  I should probably re-post this with another title, so that
--- filtered to sci.med ---
0.578  [sci.med]  Two approaches that I've used: Tofranil, 50 mg qhs, Naproxen 250mg bid.
0.547  [sci.med]  As promised, below is a personal critique of a Pressure Point Massager
0.520  [sci.med]  You certainly do not see OTC preparations advertised as such.
